# Ejercicio 12: Multimodal Embeddings con CLIP

## Objetivo de la práctica

El objetivo de este ejercicio es observar cómo un modelo multimodal como **CLIP** lleva textos e imágenes al mismo espacio vectorial.

### Actividades

1. Obtener embeddings de imágenes y textos con CLIP.
2. Comprobar que ambos tienen la misma dimensión y pueden compararse.
3. Proyectar los vectores a 2D mediante PCA.
4. Verificar emparejamientos usando similitud coseno:
   - búsqueda **texto → imagen**
   - búsqueda **imagen → texto**

> El cuaderno genera imágenes simples localmente, por lo que no necesita descargar un dataset externo. La primera ejecución sí necesita descargar el modelo CLIP desde Hugging Face.

## 1. Instalar dependencias

Ejecuta esta celda únicamente si `transformers` no está instalado. Después de instalar, puede ser necesario reiniciar el kernel.

In [ ]:
# Instalación ligera de las librerías necesarias
%pip install -q transformers pillow scikit-learn matplotlib

## 2. Importar librerías y configurar el dispositivo

In [ ]:
import numpy as np
import torch
import matplotlib.pyplot as plt

from PIL import Image, ImageDraw
from sklearn.decomposition import PCA
from sklearn.metrics.pairwise import cosine_similarity
from transformers import CLIPModel, CLIPProcessor

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Dispositivo seleccionado: {device}")

## 3. Crear un pequeño conjunto multimodal

Se generan seis imágenes de figuras geométricas y seis descripciones de texto correspondientes.

In [ ]:
def crear_imagen(figura, color, size=224):
    """Crea una imagen RGB con una figura geométrica centrada."""
    img = Image.new("RGB", (size, size), "white")
    draw = ImageDraw.Draw(img)
    margen = 42
    caja = (margen, margen, size - margen, size - margen)

    if figura == "square":
        draw.rectangle(caja, fill=color)
    elif figura == "circle":
        draw.ellipse(caja, fill=color)
    elif figura == "triangle":
        draw.polygon(
            [(size // 2, margen), (size - margen, size - margen), (margen, size - margen)],
            fill=color,
        )
    else:
        raise ValueError(f"Figura no válida: {figura}")

    return img


ejemplos = [
    ("square", "red", "a red square"),
    ("circle", "blue", "a blue circle"),
    ("triangle", "green", "a green triangle"),
    ("square", "yellow", "a yellow square"),
    ("circle", "purple", "a purple circle"),
    ("triangle", "orange", "an orange triangle"),
]

imagenes = [crear_imagen(figura, color) for figura, color, _ in ejemplos]
textos = [texto for _, _, texto in ejemplos]

fig, axes = plt.subplots(2, 3, figsize=(10, 7))
for ax, imagen, texto in zip(axes.ravel(), imagenes, textos):
    ax.imshow(imagen)
    ax.set_title(texto)
    ax.axis("off")

plt.tight_layout()
plt.show()

## 4. Cargar CLIP y obtener los embeddings

CLIP posee un codificador para imágenes y otro para textos. Sus salidas tienen la misma dimensión. Los vectores se normalizan para que su producto punto equivalga a la similitud coseno.

In [ ]:
model_name = "openai/clip-vit-base-patch32"

processor = CLIPProcessor.from_pretrained(model_name)
model = CLIPModel.from_pretrained(model_name).to(device)
model.eval()

# Procesar todas las imágenes y todos los textos en lote para mayor eficiencia
inputs = processor(
    text=textos,
    images=imagenes,
    return_tensors="pt",
    padding=True,
)
inputs = {k: v.to(device) for k, v in inputs.items()}

with torch.inference_mode():
    image_embeddings = model.get_image_features(pixel_values=inputs["pixel_values"])
    text_embeddings = model.get_text_features(
        input_ids=inputs["input_ids"],
        attention_mask=inputs["attention_mask"],
    )

# Normalización L2
image_embeddings = image_embeddings / image_embeddings.norm(dim=-1, keepdim=True)
text_embeddings = text_embeddings / text_embeddings.norm(dim=-1, keepdim=True)

# Pasar a NumPy para análisis y visualización
image_embeddings_np = image_embeddings.cpu().numpy()
text_embeddings_np = text_embeddings.cpu().numpy()

print("Forma de embeddings de imágenes:", image_embeddings_np.shape)
print("Forma de embeddings de textos:   ", text_embeddings_np.shape)
print("¿Tienen la misma dimensión?", image_embeddings_np.shape[1] == text_embeddings_np.shape[1])

## 5. Calcular la matriz de similitud coseno

In [ ]:
similaridades = cosine_similarity(text_embeddings_np, image_embeddings_np)

print("Matriz de similitud: filas = textos, columnas = imágenes")
print(np.round(similaridades, 3))

plt.figure(figsize=(8, 6))
plt.imshow(similaridades, aspect="auto")
plt.colorbar(label="Similitud coseno")
plt.xticks(range(len(textos)), [f"Img {i+1}" for i in range(len(textos))], rotation=45)
plt.yticks(range(len(textos)), textos)

for i in range(len(textos)):
    for j in range(len(textos)):
        plt.text(j, i, f"{similaridades[i, j]:.2f}", ha="center", va="center")

plt.title("Similitud entre textos e imágenes")
plt.xlabel("Imágenes")
plt.ylabel("Textos")
plt.tight_layout()
plt.show()

## 6. Búsqueda texto → imagen e imagen → texto

Para cada consulta se selecciona el elemento con la similitud coseno más alta.

In [ ]:
def buscar_imagen_por_texto(indice_texto):
    scores = similaridades[indice_texto]
    mejor_indice = int(np.argmax(scores))

    print(f"Consulta de texto: {textos[indice_texto]}")
    print(f"Mejor imagen:      imagen {mejor_indice + 1}")
    print(f"Descripción real:  {textos[mejor_indice]}")
    print(f"Similitud:         {scores[mejor_indice]:.4f}")

    plt.figure(figsize=(3, 3))
    plt.imshow(imagenes[mejor_indice])
    plt.axis("off")
    plt.title(f"Resultado: {textos[mejor_indice]}")
    plt.show()


def buscar_texto_por_imagen(indice_imagen):
    scores = similaridades[:, indice_imagen]
    mejor_indice = int(np.argmax(scores))

    print(f"Consulta: imagen {indice_imagen + 1}")
    print(f"Mejor texto: {textos[mejor_indice]}")
    print(f"Similitud:   {scores[mejor_indice]:.4f}")

    plt.figure(figsize=(3, 3))
    plt.imshow(imagenes[indice_imagen])
    plt.axis("off")
    plt.title(f"Consulta: imagen {indice_imagen + 1}")
    plt.show()


# Ejemplos
buscar_imagen_por_texto(0)
buscar_texto_por_imagen(1)

## 7. Evaluar automáticamente los emparejamientos

Como cada texto ocupa la misma posición que su imagen correcta, se calcula la exactitud Top-1 en ambas direcciones.

In [ ]:
pred_texto_a_imagen = np.argmax(similaridades, axis=1)
pred_imagen_a_texto = np.argmax(similaridades, axis=0)
objetivos = np.arange(len(textos))

accuracy_texto_a_imagen = np.mean(pred_texto_a_imagen == objetivos)
accuracy_imagen_a_texto = np.mean(pred_imagen_a_texto == objetivos)

print("Predicciones texto → imagen:", pred_texto_a_imagen + 1)
print("Predicciones imagen → texto:", pred_imagen_a_texto + 1)
print(f"Exactitud texto → imagen: {accuracy_texto_a_imagen:.2%}")
print(f"Exactitud imagen → texto: {accuracy_imagen_a_texto:.2%}")

## 8. Proyectar embeddings a dos dimensiones con PCA

Se concatenan embeddings de textos e imágenes y se aplica una única transformación PCA. Así todos los puntos permanecen en el mismo plano de comparación.

In [ ]:
todos_embeddings = np.vstack([image_embeddings_np, text_embeddings_np])
pca = PCA(n_components=2)
embeddings_2d = pca.fit_transform(todos_embeddings)

imagenes_2d = embeddings_2d[:len(imagenes)]
textos_2d = embeddings_2d[len(imagenes):]

plt.figure(figsize=(11, 8))
plt.scatter(imagenes_2d[:, 0], imagenes_2d[:, 1], marker="o", s=120, label="Imágenes")
plt.scatter(textos_2d[:, 0], textos_2d[:, 1], marker="x", s=120, label="Textos")

for i, etiqueta in enumerate(textos):
    plt.annotate(f"I{i+1}", (imagenes_2d[i, 0], imagenes_2d[i, 1]), xytext=(5, 5), textcoords="offset points")
    plt.annotate(f"T{i+1}: {etiqueta}", (textos_2d[i, 0], textos_2d[i, 1]), xytext=(5, 5), textcoords="offset points")

    # Línea que conecta cada imagen con su descripción correspondiente
    plt.plot(
        [imagenes_2d[i, 0], textos_2d[i, 0]],
        [imagenes_2d[i, 1], textos_2d[i, 1]],
        linestyle="--",
        alpha=0.4,
    )

plt.title("Embeddings multimodales CLIP proyectados con PCA")
plt.xlabel("Componente principal 1")
plt.ylabel("Componente principal 2")
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

print("Varianza explicada por las 2 componentes:", pca.explained_variance_ratio_.sum())

## 9. Conclusiones

- CLIP produce embeddings de texto e imagen con la **misma dimensión**.
- Al normalizarlos, pueden compararse directamente mediante **similitud coseno**.
- Los valores más altos de la matriz deberían corresponder a descripciones e imágenes relacionadas.
- La búsqueda cruzada permite recuperar una imagen a partir de texto y viceversa.
- PCA ayuda a visualizar el espacio común, aunque al reducir cientos de dimensiones a solo dos se pierde información.
- Los resultados pueden variar porque las imágenes geométricas son sintéticas y CLIP fue entrenado principalmente con imágenes naturales y texto de Internet.